In [1]:
import torch
import torchvision
torchvision.disable_beta_transforms_warning()
from torchvision.transforms import v2
from torchvision.models import efficientnet_b0,EfficientNet_B0_Weights,densenet121,DenseNet121_Weights
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import random
import os
import timm
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import roc_auc_score,f1_score, ConfusionMatrixDisplay
from copy import deepcopy
import csv
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class KONet(torch.nn.Module):

    def __init__(
            self,
            m1_ratio=0.4,
            m2_ratio=0.6,
            m1_dropout=0.1,
            m2_dropout=0.3,
            n_classes=2
    ):
        super().__init__()
        assert m1_ratio+m2_ratio==1
        self.n_classes=n_classes
        self.m1_ratio=m1_ratio #EfficientNet
        self.m2_ratio=m2_ratio #DenseNet
        self.m1_dropout=m1_dropout
        self.m2_dropout=m2_dropout

        self.efficient=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.efficient.classifier[0]=torch.nn.Dropout(p=self.m1_dropout,inplace=True)
        self.efficient.classifier[-1]=torch.nn.Linear(in_features=1280,out_features=self.n_classes)

        self.dense=densenet121(weights=DenseNet121_Weights.DEFAULT)
        self.dense.classifier=torch.nn.Sequential(torch.nn.Dropout(p=self.m2_dropout,inplace=True),
                                            torch.nn.Linear(in_features=1024,out_features=n_classes),
                                            )

    def forward(self, x):
        m1=self.efficient(x)
        m2=self.dense(x)
        out=self.m1_ratio*m1+self.m2_ratio*m2
        return out


def test(model,dataloader,loss_fn):
    model.eval()
    loss=0
    labels=[]
    probabilities=[]
    for data,label in dataloader:
        with torch.no_grad():
            data , label=data.to(device) , label.to(device)

            output=model(data)
            loss+=loss_fn(output , label)
            prob=output.softmax(dim=1)
            labels.append(label.detach().cpu().numpy())
            probabilities.append(prob.detach().cpu().numpy())

    labels=np.concatenate(labels,axis=0)
    probabilities=np.concatenate(probabilities,axis=0)

    loss=loss/len(dataloader)
    return loss.item(),labels,probabilities


def set_random_seed(seed: int = 2222, deterministic: bool = False):
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)  # type: ignore
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = deterministic  # type: ignore


def prep_dataset(path, image_shape=224, augmented_dataset_size=1800,
                 train_split=0.8, valid_split=0.1, test_split=0.1, seed=42):

    non_augment_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32,scale=True),
        v2.Resize((image_shape, image_shape), antialias=True),
        v2.Normalize(mean=[0.5], std=[0.5]),
    ])

    aug_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32,scale=True),
        v2.RandomAffine(degrees=30, shear=30),
        v2.RandomZoomOut(side_range=(1, 1.5)),
        v2.Resize((image_shape, image_shape), antialias=True),
        v2.Normalize(mean=[0.5], std=[0.5]),
    ])

    base_dataset = torchvision.datasets.ImageFolder(path, transform=non_augment_transform)
    augmented_dataset = torchvision.datasets.ImageFolder(path, transform=aug_transform)
    n_samples = len(base_dataset)

    labels = [s[1] for s in base_dataset.samples]
    indices = np.arange(n_samples)

    # split test first
    train_val_idx, test_idx = train_test_split(indices, test_size=test_split, stratify=labels, random_state=seed)

    kf = KFold(n_splits=10, shuffle=True, random_state=seed)
    splits = list(kf.split(train_val_idx))

    fold_datasets=[]
    factor=augmented_dataset_size//len(base_dataset)
    
    for fold, (train_indices, val_indices) in enumerate(splits):
        augmented_train_indices = np.repeat(train_indices, factor)
        train_set = torch.utils.data.Subset(augmented_dataset, augmented_train_indices)
        valid_set = torch.utils.data.Subset(base_dataset, val_indices)
        fold_datasets.append((fold+1,train_set,valid_set))

    test_subset = torch.utils.data.Subset(base_dataset, test_idx)


    return fold_datasets, test_subset

def ewc_init(ewc_model,dataloader,optimizer,loss_fn):
    model.train()
    optimizer.zero_grad()

    optpar_dict={}
    fisher_dict={}
    for train_data,train_label in tqdm(dataloader):
        train_data , train_label=train_data.to(device) , train_label.to(device)
        output=ewc_model(train_data)

        loss=loss_fn(output , train_label)
        loss.backward()

    #After iterating through dataset, save the parameters and the gradients squared in a dict
    for name , param in ewc_model.named_parameters():
        optpar_dict[name] = deepcopy(param)
        optpar_dict[name].requires_grad=False

        fisher_dict[name] = deepcopy(param.grad).pow(2)
        fisher_dict[name].requires_grad=False

    return ewc_model,optpar_dict,fisher_dict

def ewc_loss(ewc_model,optpar_dict,fisher_dict,ewc_lambda=8):
    distill_loss=0
    for name , param in ewc_model.named_parameters():
        if 'classifier' in name:
            optpar = optpar_dict[name]
            fisher = fisher_dict[name]

            distill_loss+= (fisher * (optpar - param).pow(2)).sum() * ewc_lambda
    return distill_loss

In [8]:
n_classes=2
image_shape=224
augmented_dataset_size=1800
batch_size=32
epochs=20
ewc_lambda=16
seed=42
model_name= 'mobilenet'
script_name = 'train cross dataset incremental'
results_file = "fold_results.csv"
path1="/kaggle/input/datasets/haiderazam/osteoporosis-knee-x-ray-dataset-preprocessed/Osteoporosis Knee X-ray Dataset Preprocessed"
path2="/kaggle/input/datasets/haiderazam/kod-2-class/Osteoporosis Knee X-ray Preprocessed"
weigths_path="/kaggle/input/datasets/haiderazam/cross-dataset-incremental-weights/train cross dataset incremental/model"
os.makedirs('model',exist_ok=True)

In [9]:
set_random_seed(seed)

fold_datasets1, test_set1 = prep_dataset(path1, image_shape, augmented_dataset_size)
fold_datasets2, test_set2 = prep_dataset(path2, image_shape, augmented_dataset_size)

best_acc=0
print('Model: ',model_name)
for (fold,train_set1,valid_set1),(_,train_set2,valid_set2) in zip(fold_datasets1,fold_datasets2):
    print(f"Fold {fold}")
    best_val_loss = np.inf
    best_val_acc = 0
    print(f"Train set size: {len(train_set2)}, Validation set size: {len(valid_set2)}")

    
    test_dataloader1 = DataLoader(test_set1, batch_size=batch_size, num_workers=4, pin_memory=True,
                                    persistent_workers=True, shuffle=False)
    
    
    test_dataloader2 = DataLoader(test_set2, batch_size=batch_size, num_workers=4, pin_memory=True,
                                    persistent_workers=True, shuffle=False)
    
    if 'efficient' in model_name:
        model=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        p=0.1
        model.classifier[0]=torch.nn.Dropout(p=p,inplace=True)
        model.classifier[-1]=torch.nn.Linear(in_features=1280,out_features=n_classes)
    
    elif 'dense' in model_name:
        model=densenet121(weights=DenseNet121_Weights.DEFAULT)
        p=0.3
        model.classifier=torch.nn.Sequential(torch.nn.Dropout(p=p,inplace=True),
                                            torch.nn.Linear(in_features=1024,out_features=n_classes),
                                            )
    
    elif 'conv_next' in model_name:
        p=0.3
        model=torchvision.models.convnext_tiny(weights='DEFAULT')
        model.classifier[2]=torch.nn.Sequential(torch.nn.Dropout(p=p,inplace=True),
                                            torch.nn.Linear(in_features=768,out_features=n_classes),
                                            )
    elif 'edgenext' in model_name:
        model=timm.create_model("edgenext_small.usi_in1k",
                                        pretrained=True, 
                                        features_only=False,
                                        in_chans=3,
                                        num_classes=n_classes,
                                        global_pool='avg'
                                        )
    
    elif 'KONet' in model_name:
        m1_ratio=0.6
        m2_ratio=0.4
        m1_dropout=0.1
        m2_dropout=0.3
        model=KONet(m1_ratio=m1_ratio,m2_ratio=m2_ratio,m1_dropout=m1_dropout,m2_dropout=m2_dropout,n_classes=n_classes)
    
    elif 'mobilenet' in model_name:
        model=torchvision.models.mobilenet_v3_small(weights='DEFAULT')
        model.classifier[3]=torch.nn.Linear(in_features=1024,out_features=n_classes)
    model.load_state_dict(torch.load(f'{weigths_path}/{model_name}_fold_{fold}_best_param.pkl'))
    model.to(device)
    
    loss_fn=torch.nn.CrossEntropyLoss()

    model.eval()
    test_loss1, test_labels, test_probabilities = test(model, test_dataloader1, loss_fn)
    test_pred_labels = np.argmax(test_probabilities, axis=1)
    test_acc1 = np.mean(test_pred_labels == test_labels)
    test_1_auc=roc_auc_score(test_labels,test_probabilities[:,1])
    test_1_f1=f1_score(test_labels,test_pred_labels,average='weighted')
    print(f"Test set 1 accuracy: {test_acc1}")
    print(f"F1-Score: {test_1_f1}")
    print(f"ROC_AUC: {test_1_auc}")

    test_loss2, test_labels, test_probabilities = test(model, test_dataloader2, loss_fn)
    test_pred_labels = np.argmax(test_probabilities, axis=1)
    test_acc2 = np.mean(test_pred_labels == test_labels)
    test_2_auc=roc_auc_score(test_labels,test_probabilities[:,1])
    test_2_f1=f1_score(test_labels,test_pred_labels,average='weighted')
    print(f"Test set 2 accuracy: {test_acc2}")
    print(f"F1-Score: {test_2_f1}")
    print(f"ROC_AUC: {test_2_auc}")

    harmonic_test_acc=2*test_acc1*test_acc2/(test_acc1+test_acc2)
    if best_acc<harmonic_test_acc:
        best_acc=harmonic_test_acc
        print(f"\nBest accuracy at fold {fold}")

Model:  mobilenet
Fold 1
Train set size: 1378, Validation set size: 12
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 141MB/s]


Test set 1 accuracy: 0.8157894736842105
F1-Score: 0.8146341463414635
ROC_AUC: 0.9058171745152355
Test set 2 accuracy: 0.7857142857142857
F1-Score: 0.7822765469824293
ROC_AUC: 0.8541666666666666

Best accuracy at fold 1
Fold 2
Train set size: 1378, Validation set size: 12
Test set 1 accuracy: 0.6052631578947368
F1-Score: 0.5324036095159967
ROC_AUC: 0.7008310249307479
Test set 2 accuracy: 0.6428571428571429
F1-Score: 0.5578231292517006
ROC_AUC: 0.7916666666666667
Fold 3
Train set size: 1378, Validation set size: 12
Test set 1 accuracy: 0.7368421052631579
F1-Score: 0.7368421052631579
ROC_AUC: 0.8864265927977839
Test set 2 accuracy: 0.5714285714285714
F1-Score: 0.5714285714285714
ROC_AUC: 0.6041666666666666
Fold 4
Train set size: 1378, Validation set size: 12
Test set 1 accuracy: 0.7368421052631579
F1-Score: 0.7172619047619048
ROC_AUC: 0.8116343490304709
Test set 2 accuracy: 0.7142857142857143
F1-Score: 0.6714285714285715
ROC_AUC: 0.875
Fold 5
Train set size: 1378, Validation set size: 12
